In [1]:
from configuracoes_notebooks import set_proj_dir
set_proj_dir()

O diretorio do seu projeto é coleta_cebrap
Caminho absoluto do diretorio encontrado C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap
Caminho no path.


In [2]:
from notebooks.jupyter import utils
from utils import (
    get_data_diretorio,
    check_crs,
    save_parquet_excel
)
from utils.downloads import download_malha_geosampa

# Mancha de Inundação (25 anos)

In [3]:
data_path= get_data_diretorio()

In [4]:
gdf_mancha_inundacao2=download_malha_geosampa('mancha_inundacao_25',
                                             data_path)

helloo
C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap\data\cache\mancha_inundacao_25.zip


In [5]:
gdf_mancha_inundacao2.shape

(30000, 11)

In [6]:
gdf_mancha_inundacao=download_malha_geosampa(
    'mancha_inundacao_25',
    data_path,
    True
)

Cuidado que talvez voce precise definir uma coluna de indice. A API retorna um documento com um erro e não dá status code correto!
Carregando arquivo em cache


In [7]:
gdf_mancha_inundacao.shape

(304724, 11)

In [8]:
gdf_mancha_inundacao.sample(2)

,cd_identif,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,cd_usuario,qt_tempo_r,dt_atualiz,sg_fonte_o,geometry
10864,209353,Lapa,86.60,719.955,721.33,1.38,None,25,2024-08-23,FCTH,"POLYGON ((324030.006 7398328.371, 324025.006 7..."
63482,261971,Aricanduva,21.65,733.603,733.64,0.03,None,25,2024-08-23,FCTH,"POLYGON ((343847.107 7396936.545, 343844.607 7..."


In [9]:
gdf_mancha_inundacao.columns

Index(['cd_identif', 'nm_bacia_h', 'qt_area_me', 'qt_elevaca', 'qt_cota_in',
       'qt_profund', 'cd_usuario', 'qt_tempo_r', 'dt_atualiz', 'sg_fonte_o',
       'geometry'],
      dtype='object')

Acredito que o indicador esteja se referindo apenas à área(m²) de inundação no território, mas considerando as informações de elevação, profundidade, cota de inundação* e tempo de retorno, é possível fazer análises e cálculos mais aprofundados.

*Cota de Inundação: "As cotas de inundação são valores que indicam níveis de água, que se
encontram em posições discretas sobre os rios e que são obtidas por modelos hidrodinâmicos,
estacas, sensores etc." ([ROSIM, s/d, p.2](https://files.abrhidro.org.br/Eventos/Trabalhos/154/168.pdf)).

O tempo de retorno poderia ser uma informação interessante, mas TODOS constam como 25 (anos, provavelmente), então provavelmente se refere a algo que não tempo médio de retorno da ocorrência. O mesmo vale para data de atualização.

# Padronização de nomes

In [10]:
drop_cols={
    'cd_usuario', 
    'sg_fonte_o', 
    'qt_tempo_r', 
    'dt_atualiz'
}

gdf_mancha_inundacao.rename(
    {'cd_identif':'cd_mancha_inund'}, 
    axis=1, 
    inplace=True
)
gdf_mancha_inundacao.drop(columns=drop_cols, axis=1, inplace=True)

In [11]:
gdf_mancha_inundacao.sample(2)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry
208379,406868,Tremembé,21.65,739.194,739.24,0.04,"POLYGON ((338008.694 7404486.438, 338006.194 7..."
249207,447696,Zavuvus,86.60,725.846,725.90,0.06,"POLYGON ((326791.151 7380910.942, 326786.151 7..."


# Conferir valor das geometrias

In [12]:
sum(gdf_mancha_inundacao['geometry'].isnull())

0

In [13]:
gdf_mancha_inundacao['area_mancha_inund']= (
    gdf_mancha_inundacao['geometry']
    .area
)

In [14]:
gdf_mancha_inundacao.sample(3)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,area_mancha_inund
166640,365129,Perus,86.600,820.02,822.60,2.58,"POLYGON ((324430.602 7407606.718, 324430.602 7...",86.600
108564,307053,Água Preta e Sumaré,86.600,722.78,723.96,1.18,"POLYGON ((329232.132 7397323.846, 329232.132 7...",86.600
236003,434492,Vila Leopoldina,55.424,720.85,721.57,0.72,"POLYGON ((323534.379 7395864.198, 323534.379 7...",55.424


In [15]:
sum(gdf_mancha_inundacao['area_mancha_inund'].isnull())

0

# Conferir valor da área

Vamos conferir se há alguma mancha que tenha a área cadastra igual à área da geometria:

In [16]:
(
    gdf_mancha_inundacao
    .loc[gdf_mancha_inundacao['qt_area_me']==gdf_mancha_inundacao['area_mancha_inund']]
)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,area_mancha_inund


É possível que isso aconteça devido ao arredondamento da área:

In [17]:
(
    gdf_mancha_inundacao
    .loc[gdf_mancha_inundacao['qt_area_me']==round(
        gdf_mancha_inundacao['area_mancha_inund'],
        1)
    ]
).shape

(59, 8)

Há 59 casos em que a geometria cadastrada e a da área coincidem em caso de arredondamento da área da geometria.

Agora vmaos ver se nos demais casos, as diferenças são grandes demais ou não:

In [18]:
conferir_area=gdf_mancha_inundacao.copy()

In [19]:
conferir_area['diff_area'] = (
    round(
        gdf_mancha_inundacao['area_mancha_inund'],
        1
    )-(conferir_area['qt_area_me'])
)
conferir_area.sample(2)


,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,area_mancha_inund,diff_area
2601,201090,Lapa,86.6,731.070,731.09,0.02,"POLYGON ((324968.39 7397466.72, 324963.39 7397...",86.6,-1.500013e-09
32936,231425,Água Vermelha e Lajeado,86.6,730.365,730.73,0.36,"POLYGON ((355718.024 7402069.933, 355713.024 7...",86.6,2.199997e-09


In [20]:
conferir_area = (
    conferir_area
    .loc[conferir_area['diff_area']!=0.00]
)

In [21]:
len(gdf_mancha_inundacao)-len(conferir_area)

59

In [22]:
conferir_area.sample(2)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,area_mancha_inund,diff_area
86369,284858,Aclimação,86.60,726.264,726.42,0.16,"POLYGON ((334627.966 7393850.508, 334627.966 7...",86.60,2.199997e-09
285667,484156,Cordeiro,21.65,735.091,735.58,0.48,"POLYGON ((328822.317 7385072.569, 328822.317 7...",21.65,5.000000e-02


In [23]:
conferir_area.loc[conferir_area['diff_area'] < 0, 'diff_area'] *= -1

In [24]:
conferir_area= (
    conferir_area
    .sort_values(
        by=['diff_area'], 
        ignore_index=True
    )
)

In [25]:
conferir_area['diff_area'] = round(
    conferir_area['diff_area'], 
    2
)

In [26]:
conferir_area.loc[conferir_area['diff_area']>0].shape

(132828, 9)

Depois do arredondamento da diferença das áreas, aumenta o número de 0.00, mas tmabém de outros centesimais

In [27]:
conferir_area.loc[conferir_area['diff_area']>=0.1].shape

(876, 9)

In [28]:
conferir_area.iloc[::-1].head()

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,area_mancha_inund,diff_area
304664,502860,None,6.989213e+06,0.0,0.0,0.0,"POLYGON ((341926.514 7398539.272, 341926.469 7...",18.983021,6989194.14
304663,502862,None,6.989213e+06,0.0,0.0,0.0,"POLYGON ((342016.514 7398530.612, 342014.014 7...",21.650000,6989191.54
304662,502861,None,6.989213e+06,0.0,0.0,0.0,"POLYGON ((341956.514 7398530.612, 341954.014 7...",21.650000,6989191.54
304661,502859,None,6.989213e+06,0.0,0.0,0.0,"POLYGON ((342014.014 7398552.262, 342011.514 7...",21.650000,6989191.54
304660,502864,None,6.989213e+06,0.0,0.0,0.0,"POLYGON ((341941.514 7398513.292, 341939.014 7...",21.650063,6989191.44


Ainda assim, podemos ver que há casos que a diferença permanece extremamente alta. 

Como, por padrão, estamos usando o valor da geometria, iremos manter esta regra pra este caso também:

# Conferir CRS

In [29]:
gdf_mancha_inundacao = check_crs(gdf_mancha_inundacao)

# Salvar GDF

In [30]:
save_parquet_excel(
    gdf_mancha_inundacao,
    'mancha_inundacao_25',
    data_path,
    data_subpath= 'assets'
)